<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_17_Drive_Only_Upload_Fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from decimal import Decimal
import os, json, hashlib, hmac, uuid, tempfile, sqlite3, urllib.request, zipfile, io, platform, re
import pandas as pd
import jsonschema

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass
MYDRIVE=Path(os.environ.get('ICHI2027_MYDRIVE','/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount the Google Drive used for Phases 09 and 16.'
PROJECT=MYDRIVE/'NeuroFHIR_ICHI2027'
OUT=PROJECT/'phase17_outputs'
PRIVATE=OUT/'_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True,exist_ok=True)
SHAREABLE=OUT/'ICHI2027_Phase17_SHAREABLE.json'
PINNED_FHIR_ZIP_SHA='75e5560da3cf503895a44c8ca7af17a83b4cca6c2cb5ba1883d2aec0d1cb5ac6'
FEATURES={'LHIPPOC':('hippocampus','left'),'RHIPPOC':('hippocampus','right'),
          'LENTORHIN':('entorhinal_cortex','left'),'RENTORHIN':('entorhinal_cortex','right'),
          'LMIDTEMP':('middle_temporal','left'),'RMIDTEMP':('middle_temporal','right'),
          'VENTRICLES':('ventricles','unspecified'),'ICV':('intracranial_volume','not_applicable')}
BIO_FEATURES=[f for f in FEATURES if f!='ICV']
STATES={'TRUSTED','CAUTION','REVIEW_REQUIRED'}
LONG_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1048576),b''): h.update(block)
    return h.hexdigest()

def canon(x):
    v=Decimal(str(x).strip())
    if not v.is_finite(): raise ValueError('Non-finite numeric input')
    return format(v.normalize(),'f')

def now(): return datetime.now(timezone.utc).isoformat()
print('Phase 17 initialized. All participant-level products will remain on private Google Drive.')


Mounted at /content/drive
Phase 17 initialized. All participant-level products will remain on private Google Drive.


In [2]:
public16_path=Path(os.environ.get('ICHI2027_PHASE16_JSON',str(PROJECT/'phase16_outputs'/'ICHI2027_Phase16_SHAREABLE.json')))
assert public16_path.is_file(), ('Phase16 SHAREABLE JSON missing from Google Drive. Use the Google Drive website to place it at '
    +str(PROJECT/'phase16_outputs'/'ICHI2027_Phase16_SHAREABLE.json')+'; then rerun this cell. Do not use files.upload().')
p16=json.loads(public16_path.read_text(encoding='utf-8'))
assert p16['phase']=='16_controlled_original_20b_real_replay'
assert p16['original_method']['method_changed'] is False
assert p16['original_method']['executed_exact_original_cell_indices']==list(range(19))
assert p16['verified_replay_inputs']['both_inputs_sha256_equal_phase15_private_manifest'] is True
assert p16['actual_regenerated_trace']['sidecar_granularity']=='BL_TO_M12_SUBJECT_PAIR_NOT_SCAN_LEVEL'
assert p16['actual_regenerated_trace']['participant_pair_rows_private']==258
assert p16['actual_regenerated_trace']['biological_feature_states_private']==258*7
private16_path=Path(os.environ.get('ICHI2027_PHASE16_PRIVATE',str(PROJECT/'phase16_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase16_PRIVATE_REPLAY_MANIFEST.json')))
assert private16_path.is_file(), 'The Phase16 PRIVATE manifest is required from the same Drive; rerun Notebook16 if missing.'
pr16=json.loads(private16_path.read_text(encoding='utf-8'))
assert pr16['run_utc']==p16['run_utc'], 'Phase16 private/public reports are from different runs.'
assert pr16['original_notebook_origin_sha256']==p16['original_method']['origin_notebook_sha256']
assert pr16['original_code_fingerprint']==p16['original_method']['original_code_fingerprint']
assert pr16['same_day_pair_verification']=='SOURCE_LABEL_ONLY'
sidecar_path=Path(pr16['new_private_pair_sidecar'])
if not sidecar_path.is_file():
    sidecar_path=PROJECT/'phase16_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase16_TRACE_LONGITUDINAL_JOIN_SIDECAR_PRIVATE.csv'
assert sidecar_path.is_file() and sha256_file(sidecar_path)==pr16['private_pair_sidecar_sha256'], 'Private regenerated sidecar differs from Phase16 replay.'
calib_path=PROJECT/'phase16_outputs'/'_PRIVATE_DO_NOT_SHARE'/'original_20b_replayed_outputs'/'table_locked_thresholds_bootstrap.csv'
assert calib_path.is_file() and sha256_file(calib_path)==pr16['original_calibration_csv_sha256'], 'Original calibration CSV checksum mismatch.'
print('Original Phase16 replay, regenerated sidecar and original calibration file SHA-256 verified.')

Original Phase16 replay, regenerated sidecar and original calibration file SHA-256 verified.


In [3]:
# Drive-only source discovery avoids the Google Colab browser upload error.
# If no matching source is found, upload your authorized CSV using the Google Drive WEBSITE
# (not the Colab "Choose files" dialog) to MyDrive/NeuroFHIR_ICHI2027/inputs/.
expected_source_hash=pr16['phase15_verified_original_input_hashes_PRIVATE'][LONG_NAME]
source_override=os.environ.get('ICHI2027_PHASE17_SOURCE','').strip()
options=[]
if source_override:
    options.append(Path(source_override).expanduser())
if LONG_NAME in pr16.get('source_input_paths_PRIVATE',{}):
    options.append(Path(pr16['source_input_paths_PRIVATE'][LONG_NAME]))
options.extend([Path('/content')/LONG_NAME,
                PROJECT/'inputs'/LONG_NAME,
                PROJECT/'source_data'/LONG_NAME,
                PROJECT/'phase16_outputs'/'_PRIVATE_DO_NOT_SHARE'/LONG_NAME])
# Phase16 may have used Colab's temporary upload directory, which disappears
# when a new runtime starts. Locate an existing Drive copy by exact filename.
source_path=None
seen=set()
wrong_hash_count=0

def check_exact_source(path):
    global wrong_hash_count
    path=Path(path)
    key=str(path)
    if key in seen: return None
    seen.add(key)
    if not path.is_file():return None
    if sha256_file(path)==expected_source_hash:return path
    wrong_hash_count+=1
    return None

for candidate in options:
    source_path=check_exact_source(candidate)
    if source_path is not None:break
if source_path is None:
    print('Exact source not in standard locations. Checking Google Drive for the original filename...')
    # Search the existing mounted Drive. No files are copied or uploaded.
    for candidate in MYDRIVE.rglob(LONG_NAME):
        source_path=check_exact_source(candidate)
        if source_path is not None:break
if source_path is None:
    destination=PROJECT/'inputs'/LONG_NAME
    raise FileNotFoundError(
        'The EXACT Phase16 longitudinal CSV was not found in the mounted Drive. '
        'Place your original file using the Google Drive WEBSITE at: '+str(destination)
        +' . Then rerun this cell. The browser uploader is deliberately disabled. '
        +'Existing same-name files with a different SHA-256: '+str(wrong_hash_count)
        +'. Do not replace the verified original with another version.')
assert sha256_file(source_path)==expected_source_hash, 'Source SHA-256 does not match the original.'
print('Verified original source found on Drive:',source_path)
repro=PROJECT/'_private_reproducibility'
phase09_private=repro/'phase09_private_reproducibility_manifest.json'
secret_file=repro/'ichi2027_pseudonymization_key.hex'
assert phase09_private.is_file() and secret_file.is_file(), 'Restore the original Phase09 PRIVATE manifest and HMAC key from Drive; do not create a new identity crosswalk.'
p09_private=json.loads(phase09_private.read_text(encoding='utf-8'))
assert p09_private['source_csv_sha256_private']==expected_source_hash, 'Phase09 FHIR source does not match the original TRACE real source.'
assert p09_private['fhir_schema_sha256']==PINNED_FHIR_ZIP_SHA, 'Original Phase09 used a different FHIR validation snapshot.'
KEY=bytes.fromhex(secret_file.read_text(encoding='ascii').strip())
assert len(KEY)==32, 'Invalid original Phase09 secret length.'

def opaque(*parts): return hmac.new(KEY,'|'.join(map(str,parts)).encode('utf-8'),hashlib.sha256).hexdigest()
def full_url(*parts): return 'urn:uuid:'+str(uuid.UUID(hex=opaque(*parts)[:32]))
def local_id(prefix,*parts): return prefix+'-'+opaque(*parts)[:30]
source=pd.read_csv(source_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
source.columns=[c.strip() for c in source.columns]
expected={'RID','PTID'}|{f'{tp}_{c}' for tp in ('BL','M12') for c in ('VISCODE','EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC','TEMPQC','VENTQC',*FEATURES)}
assert expected.issubset(set(source.columns)) and len(source)==258 and len(source.columns)==51
source['RID']=source['RID'].str.strip()
assert source['RID'].ne('').all() and source['RID'].is_unique
for tp in ('BL','M12'):
    source['_date_'+tp]=pd.to_datetime(source[f'{tp}_EXAMDATE'],errors='coerce').dt.date
    assert source['_date_'+tp].notna().all()
for f in FEATURES:
    for tp in ('BL','M12'):
        assert source[f'{tp}_{f}'].str.strip().ne('').all()
        for v in source[f'{tp}_{f}']:canon(v)
assert (source['_date_M12']>=source['_date_BL']).all(), 'Unexpected chronology.'
print('Exact Phase09/Phase16 longitudinal SHA-256 match and Phase09 HMAC identity crosswalk verified.')

Verified original source found on Drive: /content/TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv
Exact Phase09/Phase16 longitudinal SHA-256 match and Phase09 HMAC identity crosswalk verified.


In [4]:
sidecar=pd.read_csv(sidecar_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
sidecar.columns=[c.strip() for c in sidecar.columns]
required={'RID','BL_EXAMDATE','M12_EXAMDATE','BL_IMAGEUID','M12_IMAGEUID','BL_LONIUID','M12_LONIUID',
          'TRACE_PACKAGE_STATE','N_UNIQUE_HARD_ROOT_CAUSES','TRACE_RULE_VERSION',
          'TRACE_ORIGINAL_SOURCE_NOTEBOOK_SHA256','TRACE_REPLAY_CALIBRATION_CSV_SHA256','TRACE_STATE_GRANULARITY'}|{f'{f}_TRACE_STATE' for f in BIO_FEATURES}
assert required.issubset(set(sidecar.columns)), 'Original regenerated TRACE sidecar lacks one or more required fields.'
sidecar['RID']=sidecar['RID'].str.strip()
assert len(sidecar)==len(source)==258 and sidecar['RID'].is_unique and set(sidecar['RID'])==set(source['RID'])
assert set(sidecar['TRACE_STATE_GRANULARITY'])=={'SUBJECT_PAIR_BL_TO_M12_NOT_INDIVIDUAL_SCAN'}
assert set(sidecar['TRACE_RULE_VERSION'])=={'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16'}
assert set(sidecar['TRACE_ORIGINAL_SOURCE_NOTEBOOK_SHA256'])=={p16['original_method']['origin_notebook_sha256']}
assert set(sidecar['TRACE_REPLAY_CALIBRATION_CSV_SHA256'])=={sha256_file(calib_path)}
assert set(sidecar['TRACE_PACKAGE_STATE']).issubset(STATES)
for f in BIO_FEATURES:assert set(sidecar[f'{f}_TRACE_STATE']).issubset(STATES)
merged=source.merge(sidecar,on='RID',validate='one_to_one',suffixes=('','_TRACE'),indicator=True)
assert len(merged)==258 and merged['_merge'].eq('both').all()
for tp in ('BL','M12'):
    for c in ('EXAMDATE','IMAGEUID','LONIUID'):
        a=merged[f'{tp}_{c}'].astype(str).str.strip()
        b=merged[f'{tp}_{c}_TRACE'].astype(str).str.strip()
        if c=='EXAMDATE':
            a=pd.to_datetime(a,errors='coerce').dt.date
            b=pd.to_datetime(b,errors='coerce').dt.date
        assert a.eq(b).all(), f'Private TRACE mismatch for {tp}_{c}. Check source identity; no partial join allowed.'
for f,counts in p16['actual_regenerated_trace']['feature_level_reliability_counts_suppress_lt5'].items():
    actual=merged[f'{f}_TRACE_STATE'].value_counts()
    for state,claimed in counts.items():
        n=int(actual.get(state,0))
        assert n<5 if claimed=='<5' else n==claimed, 'State count changed since original replay.'
for state,claimed in p16['actual_regenerated_trace']['subject_pair_package_state_counts_suppress_lt5'].items():
    n=int(merged['TRACE_PACKAGE_STATE'].eq(state).sum())
    assert n<5 if claimed=='<5' else n==claimed, 'Package state counts changed.'
calib=pd.read_csv(calib_path)
assert len(calib)==8 and set(calib['feature'])==set(FEATURES)
for f,values in p16['actual_calibration']['thresholds_by_feature'].items():
    r=calib.loc[calib['feature']==f].iloc[0]
    assert abs(float(r['q90_pct'])-float(values['q90_pct']))<1e-10
    assert abs(float(r['q95_pct'])-float(values['q95_pct']))<1e-10
print('All original 258 participant-pair states and seven feature-state families reconciled to Phase16.')


All original 258 participant-pair states and seven feature-state families reconciled to Phase16.


In [5]:
FHIR_SRC_SYSTEM={
'IMAGEUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-imageuid',
'LONIUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-loniuid'}
FHIR_FIELD_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-field'
PAIR_LOCAL_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/local-trace-pair-reference'
BUNDLES=[]
FHIR_INDEX={}
LINKS=[]
NDECIMAL=0
PRIVATE_FHIR=PRIVATE/'ICHI2027_Phase17_Real_TRACE_Pair_Linked_FHIR_R4_PRIVATE.ndjson'
with PRIVATE_FHIR.open('w',encoding='utf-8') as sink:
    for _,row in merged.iterrows():
        rid=row['RID']
        person_url=full_url('Patient',rid)
        patient={'resourceType':'Patient','id':local_id('p','Patient',rid),
                 'identifier':[{'system':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/local-pseudonym',
                                'value':opaque('research-only-pseudonym',rid)}]}
        entries=[{'fullUrl':person_url,'resource':patient}]
        observation_urls={}
        for tp in ('BL','M12'):
            dt=row['_date_'+tp].isoformat()
            image_url=full_url('ImagingStudy',rid,tp)
            raw_ids=[{'system':FHIR_SRC_SYSTEM[code],'value':row[f'{tp}_{code}'].strip()}
                     for code in ('IMAGEUID','LONIUID') if row[f'{tp}_{code}'].strip()]
            img={'resourceType':'ImagingStudy','id':local_id('im','ImagingStudy',rid,tp),
                 'status':'unknown','subject':{'reference':person_url},'started':dt,'identifier':raw_ids}
            entries.append({'fullUrl':image_url,'resource':img})
            for f,(region,side) in FEATURES.items():
                v=canon(row[f'{tp}_{f}'])
                as_float=float(Decimal(v))
                assert Decimal(str(as_float))==Decimal(v), 'Numeric FHIR roundtrip changed a real value; STOP.'
                NDECIMAL+=1
                obs_url=full_url('Observation',rid,tp,f)
                anatomy=(' '.join((side,region)) if side in ('left','right') else region)
                obs={'resourceType':'Observation','id':local_id('o','Observation',rid,tp,f),
                     'status':'unknown','category':[{'coding':[{'system':'http://terminology.hl7.org/CodeSystem/observation-category','code':'imaging'}]}],
                     'identifier':[{'system':FHIR_FIELD_SYSTEM,'value':f'{tp}_{f}'}],
                     'code':{'text':anatomy+' volume'},'subject':{'reference':person_url},
                     'effectiveDateTime':dt,'derivedFrom':[{'reference':image_url}],
                     'bodySite':{'text':anatomy},'valueQuantity':{'value':as_float}}
                observation_urls[(tp,f)]=obs_url
                entries.append({'fullUrl':obs_url,'resource':obs})
        start=row['_date_BL'].isoformat()
        end=row['_date_M12'].isoformat()
        state_urls=[]
        for f in BIO_FEATURES:
            state=row[f'{f}_TRACE_STATE']
            pair_url=full_url('TRACE','FEATURE_PAIR',rid,f,'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16')
            state_urls.append(pair_url)
            assessment={'resourceType':'Observation','id':local_id('t','TRACE','FEATURE_PAIR',rid,f),
                'status':'final','identifier':[{'system':PAIR_LOCAL_SYSTEM,'value':opaque('TRACE','FEATURE_PAIR',rid,f,'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16')}],
                'code':{'text':f'Research-only TRACE Q90 longitudinal subject-pair reliability for {f}'},
                'subject':{'reference':person_url},'effectivePeriod':{'start':start,'end':end},
                'derivedFrom':[{'reference':observation_urls[('BL',f)]},
                               {'reference':observation_urls[('M12',f)]}],
                'method':{'text':'Original corrected 20B code replayed on real inputs; Q90 rule; source-labelled repeat acquisitions, not independently source-verified'},
                'valueCodeableConcept':{'text':state},
                'note':[{'text':'PAIR-LEVEL research assessment of BL-to-M12 change. Not an individual-scan reliability label; not a clinical diagnosis.'}]}
            entries.append({'fullUrl':pair_url,'resource':assessment})
            LINKS.append((opaque('private_patient_key',person_url),f,state,pair_url,
                          observation_urls[('BL',f)],observation_urls[('M12',f)],
                          p16['original_method']['origin_notebook_sha256'],sha256_file(calib_path)))
        package_url=full_url('TRACE','PACKAGE_PAIR',rid,'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16')
        package={'resourceType':'Observation','id':local_id('t','TRACE','PACKAGE_PAIR',rid),
            'status':'final','identifier':[{'system':PAIR_LOCAL_SYSTEM,'value':opaque('TRACE','PACKAGE_PAIR',rid,'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16')}],
            'code':{'text':'Research-only descriptive TRACE subject-pair package classification'},
            'subject':{'reference':person_url},'effectivePeriod':{'start':start,'end':end},
            'derivedFrom':[{'reference':u} for u in state_urls],
            'valueCodeableConcept':{'text':row['TRACE_PACKAGE_STATE']},
            'note':[{'text':'Descriptive secondary package state; derived from seven feature-pair observations, not from any single scan.'}]}
        entries.append({'fullUrl':package_url,'resource':package})
        bundle={'resourceType':'Bundle','type':'collection','entry':entries}
        assert len(entries)==27 and len({e['fullUrl'] for e in entries})==27
        BUNDLES.append(bundle)
        sink.write(json.dumps(bundle,ensure_ascii=False,allow_nan=False,separators=(',',':'))+'\n')
assert len(BUNDLES)==258 and len(LINKS)==258*7 and NDECIMAL==4128
print('PRIVATE real FHIR bundles assembled:',len(BUNDLES),'; original numeric events:',NDECIMAL,'; pair feature assessments:',len(LINKS))


PRIVATE real FHIR bundles assembled: 258 ; original numeric events: 4128 ; pair feature assessments: 1806


In [6]:
# Validate the reconstructed source and the new pair-derived resources against the exact pinned official FHIR R4 archive.
cache=PROJECT/'_private_reproducibility'/'fhir.schema.json.zip'
if cache.is_file():
    assert sha256_file(cache)==PINNED_FHIR_ZIP_SHA, 'A cached schema archive is present but differs from the pinned official ZIP.'
else:
    cache.parent.mkdir(parents=True,exist_ok=True)
    request=urllib.request.Request('https://hl7.org/fhir/R4/fhir.schema.json.zip',
                                   headers={'User-Agent':'ICHI2027-real-research-linkage/1.0'})
    try:
        with urllib.request.urlopen(request,timeout=75) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError('Pinned official FHIR R4 archive unavailable. Keep the private bundles but DO NOT claim schema validation; add the exact SHA-verified ZIP to _private_reproducibility and rerun.') from exc
    assert hashlib.sha256(raw).hexdigest()==PINNED_FHIR_ZIP_SHA, 'Downloaded official archive differs from the previously verified pinned R4 archive.'
    cache.write_bytes(raw)
assert sha256_file(cache)==PINNED_FHIR_ZIP_SHA
with zipfile.ZipFile(cache) as archive:
    name=[name for name in archive.namelist() if name.endswith('fhir.schema.json')]
    assert len(name)==1
    spec=json.loads(archive.read(name[0]))
validators={}
for typ in ('Bundle','Patient','ImagingStudy','Observation'):
    wrapper={'$schema':spec.get('$schema','http://json-schema.org/draft-06/schema#'),
             'definitions':spec['definitions'],'$ref':'#/definitions/'+typ}
    validators[typ]=jsonschema.validators.validator_for(wrapper)(wrapper)
counts=Counter()
reference_errors=Counter()
for bundle in BUNDLES:
    index={entry['fullUrl']:entry['resource'] for entry in bundle['entry']}
    for obj in [bundle]+[entry['resource'] for entry in bundle['entry']]:
        typ=obj['resourceType']
        err=next(validators[typ].iter_errors(obj),None)
        if err is not None:
            raise ValueError(f'Pinned official FHIR R4 schema failed for {typ}: validator {err.validator}; path {list(map(str,err.path))}. Participant data suppressed.')
        counts[typ]+=1
        if typ=='Observation':
            for item in obj.get('derivedFrom',[]):
                ref=item['reference']
                if ref not in index:reference_errors['unresolved_derivedFrom']+=1
                elif not (index[ref]['resourceType'] in ('Observation','ImagingStudy')):
                    reference_errors['invalid_derivedFrom_type']+=1
            if obj['subject']['reference'] not in index: reference_errors['unresolved_subject']+=1
        if typ=='ImagingStudy' and obj['subject']['reference'] not in index:
            reference_errors['unresolved_imaging_subject']+=1
assert not reference_errors, 'Original and derived FHIR referential checks failed.'
assert counts==Counter({'Bundle':258,'Patient':258,'ImagingStudy':516,'Observation':4128+1806+258})
print('Official pinned FHIR R4 validation PASS for original numeric and derived pair-level Observations; all references resolved.')


Official pinned FHIR R4 validation PASS for original numeric and derived pair-level Observations; all references resolved.


In [7]:
# This is a PRIVATE linkage-audit database, not an official OMOP / MI-CDM transformation or a new TFL algorithm.
DB=PRIVATE/'ICHI2027_Phase17_PAIR_LINKAGE_AUDIT_PRIVATE.sqlite'
if DB.exists():DB.unlink()
with sqlite3.connect(DB) as con:
    con.execute('PRAGMA foreign_keys=ON')
    con.executescript('''
    CREATE TABLE PAIR_PERSON (private_person_key TEXT PRIMARY KEY, package_state TEXT NOT NULL, package_fhir_url TEXT NOT NULL UNIQUE);
    CREATE TABLE PAIR_FEATURE (
      private_person_key TEXT NOT NULL REFERENCES PAIR_PERSON(private_person_key),
      feature TEXT NOT NULL, state TEXT NOT NULL, pair_fhir_url TEXT NOT NULL UNIQUE,
      baseline_numeric_fhir_url TEXT NOT NULL UNIQUE, month12_numeric_fhir_url TEXT NOT NULL UNIQUE,
      source_code_sha256 TEXT NOT NULL, frozen_replay_calibration_sha256 TEXT NOT NULL,
      PRIMARY KEY (private_person_key,feature));
    ''')
    rows=[]
    for _,r in merged.iterrows():
        rid=r['RID']
        rows.append((opaque('private_patient_key',full_url('Patient',rid)),r['TRACE_PACKAGE_STATE'],
                     full_url('TRACE','PACKAGE_PAIR',rid,'20B_REAL_CORRECTED_Q90_REPLAY_PHASE16')))
    con.executemany('INSERT INTO PAIR_PERSON VALUES (?,?,?)',rows)
    con.executemany('INSERT INTO PAIR_FEATURE VALUES (?,?,?,?,?,?,?,?)',LINKS)
    con.commit()
    assert not con.execute('PRAGMA foreign_key_check').fetchall()
    assert con.execute('SELECT COUNT(*) FROM PAIR_PERSON').fetchone()[0]==258
    assert con.execute('SELECT COUNT(*) FROM PAIR_FEATURE').fetchone()[0]==1806
    assert con.execute('SELECT COUNT(*) FROM PAIR_FEATURE f JOIN PAIR_PERSON p USING(private_person_key)').fetchone()[0]==1806
    assert con.execute('SELECT COUNT(DISTINCT baseline_numeric_fhir_url) FROM PAIR_FEATURE').fetchone()[0]==1806
    assert con.execute('SELECT COUNT(DISTINCT month12_numeric_fhir_url) FROM PAIR_FEATURE').fetchone()[0]==1806
    # ICV is in original numeric FHIR, but the original TRACE code does not output an ICV feature state.
    assert not con.execute("SELECT 1 FROM PAIR_FEATURE WHERE feature='ICV' LIMIT 1").fetchone()
assert sha256_file(DB)
print('Private pairing audit PASS: 258 pairs and 1,806 exactly linked 2-observation feature assessments; no ICV state invented.')


Private pairing audit PASS: 258 pairs and 1,806 exactly linked 2-observation feature assessments; no ICV state invented.


In [8]:
# Keep full private lineage and original source details outside the shareable JSON.
manifest={
  'phase':'17_real_pair_level_TRACE_to_FHIR_linkage',
  'run_utc':now(),
  'phase16_shareable_sha256':sha256_file(public16_path),
  'phase16_private_manifest_sha256':sha256_file(private16_path),
  'phase09_private_manifest_sha256':sha256_file(phase09_private),
  'longitudinal_source_sha256_PRIVATE':sha256_file(source_path),
  'phase16_original_trace_sidecar_sha256_PRIVATE':sha256_file(sidecar_path),
  'original_replay_calibration_csv_sha256_PRIVATE':sha256_file(calib_path),
  'original_source_notebook_sha256':p16['original_method']['origin_notebook_sha256'],
  'fhir_source_numeric_uri_formula':'same Phase09 private HMAC key / full_url(Patient, RID), full_url(ImagingStudy, RID, BL|M12), full_url(Observation, RID, BL|M12, feature)',
  'fhir_r4_schema_sha256':sha256_file(cache),
  'private_FHIR_NDJSON':str(PRIVATE_FHIR),
  'private_FHIR_NDJSON_sha256':sha256_file(PRIVATE_FHIR),
  'private_linkage_audit_SQLite':str(DB),
  'private_linkage_audit_SQLite_sha256':sha256_file(DB),
  'original_20b_replay_sidecar_granularity':'subject pair BL-to-M12 only',
  'no_official_micdm_or_frozen_TFL_MRI_results':True,
}
PRIVATE_MANIFEST=PRIVATE/'ICHI2027_Phase17_PRIVATE_LINKAGE_MANIFEST.json'
PRIVATE_MANIFEST.write_text(json.dumps(manifest,indent=2,allow_nan=False),encoding='utf-8')
public={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'17_real_pair_level_TRACE_to_FHIR_linkage',
 'run_utc':manifest['run_utc'],
 'status':'ACTUAL_REGENERATED_TRACE_SUBJECT_PAIR_FHIR_LINKAGE_VALIDATED_NOT_OFFICIAL_MICDM_OR_FOUR_ARM',
 'privacy':'Aggregate-only; participant-level FHIR, raw identifiers, private HMAC key, SQLite and SHA-256 of private artifacts remain on Drive.',
 'phase16_shareable_sha256':sha256_file(public16_path),
 'reused_original_20b_method':{
    'notebook_sha256':p16['original_method']['origin_notebook_sha256'],
    'code_fingerprint':p16['original_method']['original_code_fingerprint'],
    'scientific_method_changed_in_phase17':False,
    'trace_state_reference':'REGENERATED_PHASE16_NOT_HISTORICAL_FREEZE',
    'calibration_csv_revalidated_against_phase16':True,
    'repeat_acquisition_type_independently_source_verified':False},
 'exact_private_evidence_reconciliation':{
    'phase09_longitudinal_source_bytes_identical':True,
    'phase16_longitudinal_source_bytes_identical':True,
    'phase16_pair_sidecar_sha256_verified':True,
    'source_RID_date_IMAGEUID_LONIUID_join_complete':True,
    'trace_feature_state_counts_equal_phase16':True,
    'trace_package_state_counts_equal_phase16':True,
    'original_phase09_private_HMAC_key_reused':True},
 'fhir_r4':{
    'official_pinned_archive_sha256':PINNED_FHIR_ZIP_SHA,
    'official_schema_validation':'PASS',
    'validated_resource_counts':dict(counts),
    'source_numeric_volume_observation_count':NDECIMAL,
    'pair_feature_state_observation_count':len(LINKS),
    'descriptive_pair_package_observation_count':len(BUNDLES),
    'all_numeric_decimal_roundtrips_exact':True,
    'unresolved_derivedFrom_and_subject_references':sum(reference_errors.values()),
    'feature_state_derivedFrom_is_BL_and_M12_of_same_biomarker':True,
    'package_state_derivedFrom_is_seven_pair_feature_states':True,
    'source_scan_observations_have_no_invented_TRACE_state':True,
    'original_source_measurement_units_are_unset':True,
    'source_image_identifiers_are_not_claimed_DICOM_UIDs':True},
 'private_pair_linkage_audit':{
    'subject_pairs':len(BUNDLES),
    'biological_feature_states':len(LINKS),
    'pair_feature_observation_refs':2*len(LINKS),
    'ICV_pair_feature_states_in_original_method':0,
    'foreign_key_violations':0,
    'source_vs_fhir_structural_cohort_symmetric_difference':0},
 'limitations':{
    'source_repeat_pairs_two_acquisition_dates_independently_verified':False,
    'original_historical_20b_numeric_results_recovered':False,
    'exact_original_ADNI_source_export_and_units_verified':False,
    'DICOM_StudyInstanceUID_and_SeriesInstanceUID_verified':False,
    'full_Athena_snapshot_loaded':False,
    'approved_OMOP_vocabulary_and_MRI_mapping':False,
    'original_generic_TFL_v15_adapted_executed_on_real_MRI':False,
    'official_MI_CDM_load_completed':False,
    'four_arm_ETL_comparison_completed':False},
 'four_arm_comparator_scores':None,
 'official_micdm_load_results':None,
 'next_gate':'SOURCE_EVIDENCE_AND_FULL_ATHENA_THEN_IMPLEMENT_FROZEN_GENERIC_TFL_COMPARATOR_AND_OFFICIAL_MICDM_ADAPTER_WITHOUT_GUESSING_CODES'
}
assert public['fhir_r4']['official_schema_validation']=='PASS'
assert public['fhir_r4']['validated_resource_counts']['Observation']==6192
assert public['private_pair_linkage_audit']['ICV_pair_feature_states_in_original_method']==0
SHAREABLE.write_text(json.dumps(public,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print(json.dumps({'phase':public['phase'],'status':public['status'],
    'fhir_r4':public['fhir_r4']['validated_resource_counts'],
    'next_gate':public['next_gate']},indent=2))
print('Shareable JSON saved to:',SHAREABLE)


{
  "phase": "17_real_pair_level_TRACE_to_FHIR_linkage",
  "status": "ACTUAL_REGENERATED_TRACE_SUBJECT_PAIR_FHIR_LINKAGE_VALIDATED_NOT_OFFICIAL_MICDM_OR_FOUR_ARM",
  "fhir_r4": {
    "Bundle": 258,
    "Patient": 258,
    "ImagingStudy": 516,
    "Observation": 6192
  },
  "next_gate": "SOURCE_EVIDENCE_AND_FULL_ATHENA_THEN_IMPLEMENT_FROZEN_GENERIC_TFL_COMPARATOR_AND_OFFICIAL_MICDM_ADAPTER_WITHOUT_GUESSING_CODES"
}
Shareable JSON saved to: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase17_outputs/ICHI2027_Phase17_SHAREABLE.json


In [10]:
# Download through Colab's browser is intentionally disabled: its upload/download
# bridge may raise RangeError (Maximum call stack size exceeded).
# Obtain this aggregate-only report from Google Drive after the run.
assert SHAREABLE.is_file() and SHAREABLE.stat().st_size>0
print('Phase 17 SHAREABLE report saved successfully to Google Drive:')
print(SHAREABLE)
print('Download it through drive.google.com')

Phase 17 SHAREABLE report saved successfully to Google Drive:
/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase17_outputs/ICHI2027_Phase17_SHAREABLE.json
Download it through drive.google.com
